# Bagging Classifier — Complete Beginner-Friendly Program

This notebook demonstrates **Bagging (Bootstrap Aggregating)** for a classification problem using `BaggingClassifier` from scikit-learn.

We will:
1. Create a classification dataset
2. Split it into training and testing data
3. Train a normal Decision Tree
4. Train a Bagging Classifier using Decision Trees
5. Compare their performance
6. Test the trained Bagging model on new/unseen input
7. Visualize the decision boundary


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 1. Create a Dataset

For learning purposes, we create a synthetic binary classification dataset with two features: `Feature_1` and `Feature_2`.


In [ ]:
X, y = make_classification(
    n_samples=1000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42
)

df = pd.DataFrame(X, columns=['Feature_1', 'Feature_2'])
df['Target'] = y

print(df.head())
print('\nShape:', df.shape)


## 2. Visualize the Dataset


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, alpha=0.7)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Classification Dataset')
plt.show()


## 3. Train-Test Split

The model learns from the training set and is evaluated on unseen testing data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples :', len(X_test))


## 4. Baseline: Normal Decision Tree

First, we train one Decision Tree. This gives us a baseline for comparing Bagging.

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_pred)

print('Decision Tree Accuracy:', tree_accuracy)


## 5. Create the Bagging Classifier

**Bagging idea:** instead of training one Decision Tree, train many trees on different bootstrap samples of the training data and combine their predictions.

`n_estimators=100` means we train 100 base Decision Trees.


In [ ]:
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

bagging_model.fit(X_train, y_train)

bagging_pred = bagging_model.predict(X_test)
bagging_accuracy = accuracy_score(y_test, bagging_pred)

print('Bagging Accuracy:', bagging_accuracy)


## 6. Compare Decision Tree vs Bagging


In [ ]:
comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Bagging Classifier'],
    'Accuracy': [tree_accuracy, bagging_accuracy]
})

print(comparison)

comparison.plot(x='Model', y='Accuracy', kind='bar', legend=False, figsize=(7, 5))
plt.ylim(0, 1.05)
plt.ylabel('Accuracy')
plt.title('Decision Tree vs Bagging')
plt.xticks(rotation=0)
plt.show()


## 7. Classification Report

The classification report gives precision, recall, F1-score, and support.

In [ ]:
print(classification_report(y_test, bagging_pred))


## 8. Confusion Matrix


In [ ]:
cm = confusion_matrix(y_test, bagging_pred)
print('Confusion Matrix:')
print(cm)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title('Bagging Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')

plt.xticks([0, 1])
plt.yticks([0, 1])
plt.show()


## 9. Test on New / Unseen Input

Now we give the trained Bagging model completely new feature values. The model predicts whether each new sample belongs to class `0` or class `1`.

In [ ]:
new_data = np.array([
    [0.5, 1.2],
    [-1.5, -1.0],
    [2.0, 1.5],
    [-2.0, 0.5]
])

new_predictions = bagging_model.predict(new_data)
new_probabilities = bagging_model.predict_proba(new_data)

result = pd.DataFrame({
    'Feature_1': new_data[:, 0],
    'Feature_2': new_data[:, 1],
    'Predicted_Class': new_predictions,
    'Probability_Class_0': new_probabilities[:, 0],
    'Probability_Class_1': new_probabilities[:, 1]
})

print(result)


## 10. Visualize the Bagging Decision Boundary

This shows which regions of the feature space the Bagging model assigns to class 0 and class 1.

In [ ]:
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid = np.c_[xx.ravel(), yy.ravel()]
Z = bagging_model.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.25)
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k', alpha=0.7)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Bagging Classifier Decision Boundary')
plt.show()


## 11. What Bagging Is Doing

Suppose we have 100 Decision Trees:

```text
Training Data
     |
     +---- Bootstrap Sample 1 ---> Decision Tree 1 --+
     +---- Bootstrap Sample 2 ---> Decision Tree 2 --+
     +---- Bootstrap Sample 3 ---> Decision Tree 3 --+----> Voting ---> Final Prediction
     |                         ...                     |
     +---- Bootstrap Sample 100 -> Decision Tree 100-+
```

For classification, the individual trees vote for a class, and the class receiving the most votes becomes the final prediction.

### Why use Bagging?
- A single Decision Tree can have high variance and may overfit.
- Bagging trains multiple models on different bootstrap samples.
- Combining their predictions usually makes the final model more stable and less sensitive to the particular training sample.
- Random Forest is a specialized and very popular form of bagging that additionally randomizes the features considered by each tree.


## Key Parameters

| Parameter | Meaning |
|---|---|
| `estimator` | Base model used by Bagging |
| `n_estimators` | Number of base models |
| `max_samples` | Number/fraction of training samples used for each base model |
| `bootstrap=True` | Sample training rows with replacement |
| `n_jobs=-1` | Use all available CPU cores |

**Important:** Bagging is an ensemble technique. The base learner does not have to be a Decision Tree; other estimators can also be used.